User Request
    ↓
Validate Request
    ↓
ORDER PROCESSING SUBGRAPH
    ├── Check Inventory
    └── Calculate Price
    ↓
Generate Final Response

In [ ]:
from typing import TypedDict

from langgraph.graph import (
    StateGraph,
    START,
    END
)


# ============================================================
# STATE
# ============================================================

class State(TypedDict):
    product: str
    quantity: int
    inventory_status: str
    total_price: float
    final_response: str

In [ ]:
# ============================================================
# SUBGRAPH NODE 1
# ============================================================

def check_inventory(state: State):

    print("Checking inventory...")

    return {
        "inventory_status": "Available"
    }


# ============================================================
# SUBGRAPH NODE 2
# ============================================================

def calculate_price(state: State):

    print("Calculating price...")

    price_per_item = 500

    total = (
        state["quantity"]
        * price_per_item
    )

    return {
        "total_price": total
    }


# ============================================================
# CREATE SUBGRAPH
# ============================================================

subgraph_builder = StateGraph(State)

subgraph_builder.add_node(
    "check_inventory",
    check_inventory
)

subgraph_builder.add_node(
    "calculate_price",
    calculate_price
)

subgraph_builder.add_edge(
    START,
    "check_inventory"
)

subgraph_builder.add_edge(
    "check_inventory",
    "calculate_price"
)

subgraph_builder.add_edge(
    "calculate_price",
    END
)

order_subgraph = (
    subgraph_builder.compile()
)

ORDER SUBGRAPH

START
  ↓
check_inventory
  ↓
calculate_price
  ↓
END

In [ ]:
# ============================================================
# PARENT GRAPH NODE
# ============================================================

def validate_request(state: State):

    print("Validating request...")

    return {}


def final_response(state: State):

    response = (
        f"{state['quantity']} "
        f"{state['product']} ordered. "
        f"Inventory: {state['inventory_status']}. "
        f"Total Price: ₹{state['total_price']}"
    )

    return {
        "final_response": response
    }

In [ ]:
# ============================================================
# PARENT GRAPH
# ============================================================

builder = StateGraph(State)


builder.add_node(
    "validate_request",
    validate_request
)


# Entire subgraph becomes one node
builder.add_node(
    "order_processing",
    order_subgraph
)


builder.add_node(
    "final_response",
    final_response
)


builder.add_edge(
    START,
    "validate_request"
)

builder.add_edge(
    "validate_request",
    "order_processing"
)

builder.add_edge(
    "order_processing",
    "final_response"
)

builder.add_edge(
    "final_response",
    END
)


graph = builder.compile()

In [ ]:
result = graph.invoke(
    {
        "product": "Keyboard",
        "quantity": 2,
        "inventory_status": "",
        "total_price": 0,
        "final_response": ""
    }
)

print(
    result["final_response"]
)

In [ ]:
# ============================================================
# LANGGRAPH SUBGRAPH + SEND API
# COMPLETE END-TO-END EXAMPLE
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

from typing import TypedDict, Annotated
import operator

from langgraph.graph import (
    StateGraph,
    START,
    END
)

from langgraph.types import Send

from langchain_openai import ChatOpenAI


# ============================================================
# MODEL
# ============================================================

model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)


# ============================================================
# PARENT GRAPH STATE
# ============================================================

class ParentState(TypedDict):

    # Topics provided by user
    topics: list[str]

    # Results coming from parallel workers
    # operator.add combines results from multiple branches
    research_results: Annotated[
        list[str],
        operator.add
    ]

    # Final combined response
    final_answer: str


# ============================================================
# SUBGRAPH STATE
# ============================================================

class ResearchState(TypedDict):

    topic: str

    research: str

    summary: str


# ============================================================
# SUBGRAPH NODE 1
# RESEARCH TOPIC
# ============================================================

def research_topic(state: ResearchState):

    topic = state["topic"]

    print(
        f"\n[SUBGRAPH] Researching topic: {topic}"
    )

    response = model.invoke(
        f"""
You are an AI researcher.

Explain the following Generative AI topic:

Topic:
{topic}

Provide:

1. Definition
2. Important concepts
3. One practical use case

Keep the explanation concise.
"""
    )

    return {
        "research": response.content
    }


# ============================================================
# SUBGRAPH NODE 2
# SUMMARIZE RESEARCH
# ============================================================

def summarize_topic(state: ResearchState):

    print(
        f"[SUBGRAPH] Summarizing topic: {state['topic']}"
    )

    response = model.invoke(
        f"""
Create a concise summary of the following research.

Topic:
{state['topic']}

Research:

{state['research']}
"""
    )

    return {
        "summary": response.content
    }


# ============================================================
# BUILD RESEARCH SUBGRAPH
# ============================================================

research_builder = StateGraph(
    ResearchState
)


research_builder.add_node(
    "research_topic",
    research_topic
)


research_builder.add_node(
    "summarize_topic",
    summarize_topic
)


research_builder.add_edge(
    START,
    "research_topic"
)


research_builder.add_edge(
    "research_topic",
    "summarize_topic"
)


research_builder.add_edge(
    "summarize_topic",
    END
)


# Compile Subgraph
research_subgraph = (
    research_builder.compile()
)


# ============================================================
# PARENT GRAPH NODE
# PREPARE TOPICS
# ============================================================

def prepare_topics(state: ParentState):

    print("\n" + "=" * 60)
    print("PARENT GRAPH")
    print("=" * 60)

    print("\nTopics received:")

    for topic in state["topics"]:

        print(
            "-",
            topic
        )

    return {}


# ============================================================
# WORKER
#
# Each worker receives ONE topic.
# It then invokes the Research Subgraph.
# ============================================================

def research_worker(state):

    topic = state["topic"]

    print(
        f"\n[WORKER] Starting subgraph for: {topic}"
    )


    # Invoke the subgraph
    result = research_subgraph.invoke(
        {
            "topic": topic,
            "research": "",
            "summary": ""
        }
    )


    print(
        f"[WORKER] Completed: {topic}"
    )


    # Return result to parent graph
    return {

        "research_results": [

            f"""
TOPIC: {topic}

{result["summary"]}
"""
        ]
    }


# ============================================================
# SEND API
#
# Dynamically create one worker execution
# for every topic.
# ============================================================

def distribute_topics(state: ParentState):

    print(
        "\n[PARENT] Distributing topics using Send API..."
    )


    return [

        Send(
            "research_worker",
            {
                "topic": topic
            }
        )

        for topic in state["topics"]
    ]


# ============================================================
# COMBINE ALL RESULTS
# ============================================================

def combine_results(state: ParentState):

    print(
        "\n[PARENT] Combining all research results..."
    )


    combined_research = "\n\n".join(
        state["research_results"]
    )


    response = model.invoke(
        f"""
You are an AI instructor.

Combine the following research summaries into one
well-structured explanation.

Keep each topic clearly separated.

Research summaries:

{combined_research}
"""
    )


    return {
        "final_answer": response.content
    }


# ============================================================
# BUILD PARENT GRAPH
# ============================================================

builder = StateGraph(
    ParentState
)


# Add nodes
builder.add_node(
    "prepare_topics",
    prepare_topics
)


builder.add_node(
    "research_worker",
    research_worker
)


builder.add_node(
    "combine_results",
    combine_results
)


# ============================================================
# EDGES
# ============================================================

builder.add_edge(
    START,
    "prepare_topics"
)


# Dynamic fan-out
# distribute_topics() returns multiple Send objects
builder.add_conditional_edges(
    "prepare_topics",
    distribute_topics
)


# All workers eventually move here
builder.add_edge(
    "research_worker",
    "combine_results"
)


builder.add_edge(
    "combine_results",
    END
)


# ============================================================
# COMPILE PARENT GRAPH
# ============================================================

graph = builder.compile()


# ============================================================
# RUN GRAPH
# ============================================================

result = graph.invoke(
    {
        "topics": [
            "Retrieval-Augmented Generation",
            "AI Agents",
            "Memory Management"
        ],

        "research_results": [],

        "final_answer": ""
    }
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print()

print(
    result["final_answer"]
)